In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform

Compose(
    ToTensor()
    Normalize(mean=(0.5,), std=(0.5,))
)

In [4]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_dataset

100%|██████████| 9.91M/9.91M [01:51<00:00, 89.3kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 47.0kB/s]
100%|██████████| 1.65M/1.65M [00:08<00:00, 192kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.1kB/s]


Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [5]:
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [7]:
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


In [8]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [9]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = torch.argmax(outputs, dim=1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )


Epoch [1/5] Loss: 0.1627 Accuracy: 95.06%
Epoch [2/5] Loss: 0.0468 Accuracy: 98.57%
Epoch [3/5] Loss: 0.0322 Accuracy: 99.02%
Epoch [4/5] Loss: 0.0237 Accuracy: 99.22%
Epoch [5/5] Loss: 0.0177 Accuracy: 99.45%


In [10]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predicted = torch.argmax(outputs, dim=1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

In [11]:
test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 98.93%
